In [1]:
!pip install torch==2.9.0 transformers==4.57.3 pandas==2.2.2 numpy==2.0.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F
import pandas as pd

In [3]:
# Model Setup
model_name = "Qwen/Qwen2.5-0.5B"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name, dtype=dtype, trust_remote_code=True
).to(device)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if model.config.pad_token_id is None:
    model.config.pad_token_id = tokenizer.eos_token_id

print("Model Loaded!\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model Loaded!



In [ ]:
def get_top_n_predictions(logits, n=5):
    probs = F.softmax(logits, dim=-1) # last dim which is Vocab.
    top_probs, top_ids = torch.topk(probs, n)
    return [(tokenizer.decode([int(i)]), float(p)) for i, p in zip(top_ids, top_probs)]

def visualize_step(step, text, preds, chosen):
    print("\n" + "=" * 80)
    print(f"STEP {step}")
    print(f"Current text: \"{text}\"")
    print("\nTop-n-next-token prediction:")

    df = pd.DataFrame({
        "Rank": range(1, 6),
        "Token": [repr(t) for t, _ in preds],
        "Probability": [f"{p:.4f}" for _, p in preds]
    })
    print(df.to_string(index=False))
    print(f"\nSelected token: {repr(chosen)}")

In [ ]:
def generate_step_by_step(
        prompt,
        steps=5
):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    # Prompt -> [[ 785, 8251, 7578,  389,  279]] --> Shape of (1, 5)
    # i.e. 1 batch 5 tokens [batch size, seq_len]
    print(input_ids, input_ids.shape)
    current_text = prompt
    print("=" * 80)
    print("AUTOREGRESSIVE GENERATION (GREEDY)")
    print("=" * 80)
    print(f"Prompt: \"{prompt}\"")

    for step in range(steps):
        with torch.no_grad():# Runs Forward pass
            # model(input_ids) --> (batch_size, seq_len, vocab_size) : (1, 5, V)
            # batch -> 0 
            # -? Input (1, 5, 768) - 768 dim vector : After embedding layer
            # (batch=1, seq_len=5, hidden_dim=768)
            # Self Attention - ffn layer -> (1, 5, 768)
            # Final Layer = logits = W@X +b , Here W - (Vocab Size, 768), b - (Vocab Size)
            # Shaape of X - (768, )  This is after extract last pos [0, -1]
            # Hence final layer -> (Vocab Size, )
            logits = model(input_ids).logits[0, -1] # extract last token's logits
            # as last token's position has full context. Causality
            # information Odering 

        next_id = torch.argmax(logits).view(1, 1) # find index with max logit value
        # view reshapes to (1, 1) for concatenations, Reshapes [[next id]]
        display_logits = logits

        preds = get_top_n_predictions(display_logits)
        token_str = tokenizer.decode(next_id[0]) 
        # extracts the token id and convert to string

        visualize_step(step + 1, current_text, preds, token_str)

        input_ids = torch.cat([input_ids, next_id], dim=-1)
        current_text += token_str

    print("\nFINAL OUTPUT:")
    print(current_text)
    print("=" * 80 + "\n")

# Greedy
generate_step_by_step(
    "The cat sat on the",
    steps=5,
)

tensor([[ 785, 8251, 7578,  389,  279]]) torch.Size([1, 5])
AUTOREGRESSIVE GENERATION (GREEDY)
Prompt: "The cat sat on the"

STEP 1
Current text: "The cat sat on the"

Top-n-next-token prediction:
 Rank     Token Probability
    1    ' mat'      0.8426
    2    ' cat'      0.0247
    3  ' fence'      0.0110
    4      '\n'      0.0094
    5 ' window'      0.0046

Selected token: ' mat'

STEP 2
Current text: "The cat sat on the mat"

Top-n-next-token prediction:
 Rank   Token Probability
    1     '.'      0.2193
    2   '.\n'      0.1617
    3     ','      0.1018
    4  ' and'      0.0855
    5 '.\n\n'      0.0657

Selected token: '.'

STEP 3
Current text: "The cat sat on the mat."

Top-n-next-token prediction:
 Rank   Token Probability
    1  ' The'      0.1806
    2     ' '      0.1752
    3 ' What'      0.0722
    4    ' ('      0.0445
    5   ' It'      0.0346

Selected token: ' The'

STEP 4
Current text: "The cat sat on the mat. The"

Top-n-next-token prediction:
 Rank   Token Pro